# Übung 2b: Dieselben Daten ohne Labels – Anomalieerkennung
**Internet of Production – KI, Woche 1** · ca. 20 min · Arbeit in Zweiergruppen

**Roter Faden:** In Übung 2 haben Sie einen Klassifikator mit **Labels** trainiert („Blockade innerhalb von 6 Stunden“). In der Praxis sind solche Labels selten: Ein neuer Drucker hat keine Ausfallhistorie, und Ausfälle sind wenige. Hier nutzen Sie **nur das normale Verhalten** des Druckers und melden alles, was **davon abweicht**.

**Aufgabe:** Lernen Sie aus den ersten 70 % der Daten, was normal ist, ohne das Label `y` zu verwenden. Lösen Sie einen Alarm aus, wenn die Energie der letzten zwei Stunden ungewöhnlich hoch ist. Das Label dient nur zur **Bewertung** der Alarme.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

DATA = Path('../../00_Data/printfarm')
HORIZON = 360                       # minutes
FEATURES = ['E30', 'E120', 'slope']

def make_blockage_data(k, horizon=HORIZON):
    """The pipeline of exercise 2 in one function: one row per 10th printing minute."""
    d = pd.read_csv(DATA / f'Printer_{k}.csv', sep=';', decimal=',', low_memory=False).dropna(how='all')
    d['Timestamp'] = pd.to_datetime(d['Timestamp'], format='%d.%m.%Y %H:%M')
    d = d.set_index('Timestamp')
    d['Energy'] = d['Total_Power'].diff()
    e = pd.read_csv(DATA / f'Printer_{k}_error.csv', sep=';').dropna(how='all')
    e['Timestamp'] = pd.to_datetime(e['Timestamp'], format='%d.%m.%Y %H:%M')
    blockages = pd.to_datetime(np.sort(e.loc[e['Error'] == 'Extruder blocked', 'Timestamp'].values))
    p = d[d['State'] == 'printing'].copy()
    p['E'] = p['Energy'] * 1000                              # Wh per minute
    idx = p.index.values
    nxt = np.searchsorted(blockages.values, idx)
    mtb = np.where(nxt < len(blockages), (blockages.values[np.minimum(nxt, len(blockages) - 1)] - idx) / np.timedelta64(1, 'm'), np.inf)
    p['y'] = ((mtb > 0) & (mtb <= horizon)).astype(int)      # only used to EVALUATE here
    p['E30'] = p['E'].rolling(30).mean()
    p['E120'] = p['E'].rolling(120).mean()
    p['slope'] = p['E120'] - p['E120'].shift(240)
    return p.dropna(subset=FEATURES).iloc[::10], blockages

def evaluate_alarms(alarm, test, blockages):
    """alarm: boolean Series on the index of `test`. Counts blockages announced within 6 h and false alarm episodes per week."""
    t0, t1 = test.index[0], test.index[-1]
    blk = [t for t in blockages if t0 <= t <= t1]
    announced = sum(bool(alarm[(alarm.index >= t - pd.Timedelta(hours=6)) & (alarm.index <= t)].any()) for t in blk)
    times = alarm[alarm].index
    weeks = (t1 - t0).days / 7
    if len(times) == 0:
        return {'announced': f'0/{len(blk)}', 'false episodes / week': 0.0}
    new_episode = np.concatenate([[0], np.diff(times.values) > np.timedelta64(6, 'h')])   # a gap of more than 6 h starts a new episode
    ep = pd.DataFrame({'ep': np.cumsum(new_episode), 'y': test.loc[times, 'y'].values}).groupby('ep')['y'].max()
    return {'announced': f'{announced}/{len(blk)}', 'false episodes / week': round(float((ep == 0).sum() / weeks), 2)}

data, blockages = make_blockage_data(1)
cut = int(len(data) * 0.7)
reference, test = data.iloc[:cut], data.iloc[cut:]          # reference = 'what is normal', test = the future
print(len(reference), 'reference samples,', len(test), 'test samples,', len([t for t in blockages if t >= test.index[0]]), 'blockages in the test period')

## Schritt 1: Was ist normal?
Berechnen Sie Mittelwert und Standardabweichung von `E120` im **Referenzzeitraum** und standardisieren Sie: `z = (E120 - Mittelwert) / Standardabweichung`. Stellen Sie den Testzeitraum wie eine **Regelkarte** dar (wie in der statistischen Prozesskontrolle) und markieren Sie die Extruder-Blockaden.

In [ ]:
mu, sd = reference['E120'].___(), reference['E120'].___()
z_ref = (reference['E120'] - mu) / sd
z_test = (test['E120'] - mu) / sd
print(f'normal E120: {mu:.3f} Wh/min, standard deviation {sd:.3f}')

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(z_test.index, z_test.values, '.', ms=1.5, color='#009EE0')
for t in [t for t in blockages if t >= test.index[0]]:
    ax.axvline(t, color='#E2001A', lw=1)
ax.axhline(3, color='#EE7F00', ls='--'); ax.set_ylabel('z of E120'); ax.set_title('test period; red lines = extruder blockages'); plt.show()

**Was sehen Sie vor den roten Linien? Warum springt die Kurve nicht einfach über die Grenze?**

*Ihre Antwort:*

## Schritt 2: Die Grenze aus den Normaldaten setzen
Die Grenze ist ein **Quantil der Referenzdaten**: Beim 95-%-Quantil würden 5 % der normalen Stichproben gemeldet. Zählen Sie dann, wie viele Blockaden angekündigt werden und wie viele **Fehlalarm-Episoden je Woche** auftreten.

In [ ]:
limit = np.quantile(z_ref, ___)
alarm = z_test > limit
print(f'limit z > {limit:.2f}  (E120 > {mu + limit * sd:.3f} Wh/min)')
print(evaluate_alarms(alarm, test, blockages))

## Schritt 3: Die Grenze verschieben
Die Grenze steuert die Zahl der Fehlalarme. Probieren Sie mehrere Quantile.

In [ ]:
rows = []
for q in [0.90, 0.95, 0.98, ___]:
    lim = np.quantile(z_ref, q)
    r = evaluate_alarms(z_test > lim, test, blockages)
    rows.append({'quantile': q, 'limit (z)': round(lim, 2), **r})
pd.DataFrame(rows).set_index('quantile')

**Welche Grenze wählen Sie für einen Techniker, der pro Tag einen Alarm prüfen kann? Welche für eine Anlage, in der ein Stillstand 10 Stunden kostet?**

*Ihre Antwort:*

## Schritt 4: Vergleich mit dem Klassifikator aus Übung 2
Trainieren Sie das KNN aus Übung 2 (k = 25, skalierte Merkmale) **mit Labels** auf demselben Referenzzeitraum und bewerten Sie seine Warnungen auf dieselbe Weise.

In [ ]:
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=___)).fit(reference[FEATURES], reference['y'])
knn_alarm = pd.Series(knn.predict(test[FEATURES]) == 1, index=test.index)
rows = [
    {'method': 'z-value of E120 (no labels)', 'needs labels': 'no', 'AUC': roc_auc_score(test['y'], z_test), **evaluate_alarms(z_test > np.quantile(z_ref, 0.95), test, blockages)},
    {'method': 'KNN k = 25 (with labels)', 'needs labels': 'yes', 'AUC': roc_auc_score(test['y'], knn.predict_proba(test[FEATURES])[:, 1]), **evaluate_alarms(knn_alarm, test, blockages)},
]
pd.DataFrame(rows).set_index('method').round(3)

Die ROC-Kurven zeigen die **Rangordnungsqualität** beider Scores über alle Schwellen (die AUC). Wie viele Fehlalarme eine Grenze je Woche verursacht, sagen sie nicht: Dafür gilt die Tabelle oben.

In [ ]:
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(7, 5))
fpr, tpr, _ = roc_curve(test['y'], z_test)
plt.plot(fpr, tpr, label=f'z-value (AUC = {auc(fpr, tpr):.2f})')
fpr, tpr, _ = roc_curve(test['y'], knn.predict_proba(test[FEATURES])[:, 1])
plt.plot(fpr, tpr, label=f'KNN k = 25 (AUC = {auc(fpr, tpr):.2f})')
plt.plot([0, 1], [0, 1], 'r--', label='random guess')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curves of the two scores')
plt.legend()
plt.show()

**Hilft das Label hier? Wann würden Sie den Klassifikator vorziehen, wann den Anomalie-Detektor?**

*Ihre Antwort:*

## Schritt 5 (optional, bei Zeit): Ein komplexerer Detektor
Der **Isolation Forest** nutzt alle drei Merkmale und lernt, was sich *leicht isolieren* lässt (ungewöhnlich ist). Trainieren Sie ihn auf den Referenzdaten, setzen Sie die Grenze wieder aus der Referenz und vergleichen Sie ihn mit dem einfachen z-Wert (AUC des Scores und Alarme).

In [ ]:
iso = IsolationForest(n_estimators=200, random_state=0).fit(reference[___])
s_ref = pd.Series(-iso.score_samples(reference[FEATURES]), index=reference.index)    # higher = more unusual
s_test = pd.Series(-iso.score_samples(test[FEATURES]), index=test.index)
lim_iso = np.quantile(s_ref, 0.95)

print('AUC z-value of E120 :', round(roc_auc_score(test['y'], z_test), 3), evaluate_alarms(z_test > np.quantile(z_ref, 0.95), test, blockages))
print('AUC Isolation Forest:', round(roc_auc_score(test['y'], s_test), 3), evaluate_alarms(s_test > lim_iso, test, blockages))

**Gewinnt der komplexere Isolation Forest? Warum (nicht)?**

*Ihre Antwort:*

## Schritt 6 (optional, wandert in Woche 3): Ein neuer Drucker
Wenden Sie die Methode auf **Drucker 2** an (ein Drucker mit 18 Blockaden), und zwar auf zwei Arten: (a) mit der Referenz von **Drucker 2** selbst (seine ersten 70 %), (b) mit der Referenz von **Drucker 1**. Worin unterscheiden sich die Ergebnisse?

In [ ]:
data2, blockages2 = make_blockage_data(___)
cut2 = int(len(data2) * 0.7)
ref2, test2 = data2.iloc[:cut2], data2.iloc[cut2:]
mu2, sd2 = ref2['E120'].mean(), ref2['E120'].std()
print(f'normal E120: printer 1 {mu:.3f} (sd {sd:.3f}), printer 2 {mu2:.3f} (sd {sd2:.3f})')

z_own = (test2['E120'] - mu2) / sd2
lim_own = np.quantile((ref2['E120'] - mu2) / sd2, 0.95)
z_foreign = (test2['E120'] - mu) / sd                      # baseline of printer 1
print('own baseline    :', round(roc_auc_score(test2['y'], z_own), 3), evaluate_alarms(z_own > lim_own, test2, blockages2))
print('baseline of P1  :', round(roc_auc_score(test2['y'], z_foreign), 3), evaluate_alarms(z_foreign > np.quantile(z_ref, 0.95), test2, blockages2))

**Was braucht ein Drucker, bevor der Anomalie-Detektor auf ihn angewendet werden kann? Was bedeutet das für eine Flotte von 30 Druckern?**

*Ihre Antwort:*